In [1]:
!pip install pennylane pennylane-lightning-gpu --upgrade
!pip install custatevec-cu12 cuquantum-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 M

# Imports

In [20]:
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import time

# =========================
# CONFIG
# =========================
torch.manual_seed(42)

n_qubits = 6        # increase for stronger quantum effect
n_layers = 3
input_dim = 16
hidden_dim = 32
vocab_size = 10
batch_size = 16
epochs = 5

# =========================
# DEVICE (Lightning GPU)
# =========================
dev = qml.device("lightning.gpu", wires=n_qubits)

# QRL

In [21]:
@qml.qnode(dev, interface="torch", diff_method="adjoint")
def qrl_circuit(inputs, weights):

    # Encoding
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    # Variational layers
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)

        # Entanglement
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])

    # IMPORTANT: return LIST (not tuple)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Convert QNode → Torch Layer

In [22]:
weight_shapes = {"weights": (n_layers, n_qubits, 2)}
qlayer = qml.qnn.TorchLayer(qrl_circuit, weight_shapes)

# Classical Baseline Model

In [23]:
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        return self.net(x)

# Hybrid Quantum Model

In [24]:
class HybridQuantumModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_qubits)
        )

        self.q_layer = qlayer
        self.decoder = nn.Linear(n_qubits, vocab_size)

    def forward(self, x):

        # Classical encoding
        h = self.encoder(x)
        h = torch.tanh(h)

        # FIX: manual batching
        q_out = torch.stack([self.q_layer(h_i) for h_i in h])

        logits = self.decoder(q_out)
        return logits

# Dataset

In [25]:
x = torch.randn(512, input_dim)
y = torch.randint(0, vocab_size, (512,))

dataset = TensorDataset(x, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training Loop

In [26]:
def train(model, dataloader, optimizer, loss_fn, epochs):

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for x, y in pbar:

            optimizer.zero_grad()

            logits = model(x)
            loss = loss_fn(logits, y)

            loss.backward()

            grad_norm = sum(
                p.grad.norm().item()
                for p in model.parameters()
                if p.grad is not None
            )

            optimizer.step()

            epoch_loss += loss.item()

            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "grad": f"{grad_norm:.4f}"
            })

        print(f"\nEpoch {epoch+1} Avg Loss: {epoch_loss/len(dataloader):.4f}")

# Evaluation Function

In [27]:
def evaluate(model, dataloader):

    model.eval()

    correct = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for x, y in dataloader:
            logits = model(x)
            preds = logits.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    end_time = time.time()

    accuracy = correct / total
    time_per_sample = (end_time - start_time) / total

    return accuracy, time_per_sample

# Run

In [28]:
classical_model = ClassicalModel()
quantum_model = HybridQuantumModel()

opt_classical = torch.optim.Adam(classical_model.parameters(), lr=1e-3)
opt_quantum = torch.optim.Adam(quantum_model.parameters(), lr=1e-3)

loss_fn = nn.CrossEntropyLoss()

# Train Classical
print("\n=== Training Classical Model ===")
train(classical_model, dataloader, opt_classical, loss_fn, epochs)

# Train Quantum
print("\n=== Training Quantum Model ===")
train(quantum_model, dataloader, opt_quantum, loss_fn, epochs)

# Evaluate
acc_c, time_c = evaluate(classical_model, dataloader)
acc_q, time_q = evaluate(quantum_model, dataloader)

print("\n=== FINAL RESULTS ===")
print(f"Classical → Accuracy: {acc_c:.4f}, Time/sample: {time_c:.6f}")
print(f"Quantum   → Accuracy: {acc_q:.4f}, Time/sample: {time_q:.6f}")


=== Training Classical Model ===


Epoch 1/5: 100%|██████████| 32/32 [00:00<00:00, 343.66it/s, loss=2.3827, grad=1.2874]



Epoch 1 Avg Loss: 2.3294


Epoch 2/5: 100%|██████████| 32/32 [00:00<00:00, 354.00it/s, loss=2.3509, grad=1.2407]



Epoch 2 Avg Loss: 2.3054


...


Epoch 3 Avg Loss: 2.3236


Epoch 4/5: 100%|██████████| 32/32 [00:16<00:00,  1.90it/s, loss=2.2403, grad=0.7757]



Epoch 4 Avg Loss: 2.3188


Epoch 5/5: 100%|██████████| 32/32 [00:17<00:00,  1.85it/s, loss=2.3575, grad=0.4344]



Epoch 5 Avg Loss: 2.3137

=== FINAL RESULTS ===
Classical → Accuracy: 0.1602, Time/sample: 0.000024
Quantum   → Accuracy: 0.0957, Time/sample: 0.028262
